In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.encode import AutoQueryEncoder
from pyserini.search import get_topics, get_qrels
from tqdm import tqdm

/Users/winstondong/miniforge3/envs/adnlp_hyde_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialize Contriever Index and Query Encoder
We use [Pyserini](https://github.com/castorini/pyserini) as the search interface for the experiment. Please following the guidance in Pyserini to create Contriever index using the checkpoint from original Contriever work.

In [2]:
query_encoder = AutoQueryEncoder(encoder_dir='facebook/contriever', pooling='mean', use_fp16=False)
searcher = FaissSearcher('contriever_msmarco_index/', query_encoder)
corpus = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')

Mar 31, 2026 4:28:16 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


### Load query and judegments for dl19-passage dataset

In [3]:
topics = get_topics('dl19-passage')
qrels = get_qrels('dl19-passage')

## Run Contriever

In [4]:
with open('dl19-contriever-top1000-trec', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

100%|██████████| 43/43 [00:25<00:00,  1.72it/s]


In [1]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage dl19-contriever-top1000-trec

map                   	all	0.2399
ndcg_cut_10           	all	0.4454
recall_1000           	all	0.7459


In [ ]:
import openai
import os

KEY = 'YOUR_API_KEY'
client = openai.OpenAI(api_key=KEY)

def call_codex_read_api(prompt: str):
    def parse_api_result(result):
        to_return = []
        for choice in result.choices:
            text = choice.message.content
            if choice.logprobs and choice.logprobs.content:
                logprob = sum(token.logprob for token in choice.logprobs.content)
            else:
                logprob = 0
            to_return.append((text, logprob))
        return [r[0] for r in sorted(to_return, key=lambda tup: tup[1], reverse=True)]

    result = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=512,
        temperature=0.7,
        top_p=1,
        n=8,
        stop=['\n\n\n'],
        logprobs=True,
    )
    return parse_api_result(result)

## Run HyDE

In [4]:
from time import sleep
import numpy as np
import json
from tqdm import tqdm
with open('hyde-dl19-contriever-gpt3-top1000-8rep-trec', 'w') as f, open('hyde-dl19-gpt3-gen.jsonl', 'w') as fgen:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            print(query)
            prompt = f"""Please write a passage to answer the question
Question: {query}
Passage:"""
            get_result = False
            while not get_result:
                try:
                    contexts = [c.strip() for c in call_codex_read_api(prompt)] + [query]
                    all_emb_c = []
                    for c in contexts:
                        c_emb = query_encoder.encode(c)
                        all_emb_c.append(np.array(c_emb))
                    all_emb_c = np.array(all_emb_c)
                    avg_emb_c = np.mean(all_emb_c, axis=0)
                    avg_emb_c = avg_emb_c.reshape((1, len(avg_emb_c)))
                    fgen.write(json.dumps({'query_id': qid, 'query': query, 'contexts': contexts})+'\n')
                    get_result = True
                except:
                    sleep(1)
            print(contexts)
            hits = searcher.search(avg_emb_c, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

NameError: name 'topics' is not defined

In [ ]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage hyde-dl19-contriever-gpt3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage hyde-dl19-contriever-gpt3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage hyde-dl19-contriever-gpt3-top1000-8rep-trec
